## Homework

---


##### Assignment:


Зробити аналіз датасета. Побудувати не менше 10 зображень, графіків, діаграм.

По кожній картинці зробити невелике резюме.

Створити не менше 5 нових фічей вручну

Додатково - застосувати бібліотеку для автоматичної генерації нових фічей


##### Feature Descriptions:


**player_data.csv**

- name
- player name
- year_start
- year_end
- position
- height
- weight
- birth_date
- college

**players.csv**

- Player - Player's full name (first and last)
- height - Height in cm
- weight - Weight in kg
- collage - The college said player attended
- born - Birthdate of the player
- birth_city - City where the player was born.
- birth_state - State where the player was born

**seasons_stats.csv**

- Year - Season
- Player - name
- Pos - Position
- Age - Age
- Tm - Team
- G - Games
- GS - Games Started
- MP - Minutes Played
- PER - Player Efficiency Rating
- TS% - True Shooting %
- 3Par - 3-Point Attempt Rate
- FTr - Free Throw Rate
- ORB% - Offensive Rebound Percentage
- DRB% - Defensive Rebound Percentage
- TRB% - Total Rebound Percentage
- AST% - Assist Percentage
- STL% - Steal Percentage
- BLK% - Block Percentage
- TOV% - Turnover Percentage
- USG% - Usage Percentage
- blanl - empty
- OWS - Offensive Win Shares
- DWS - Defensive Win Shares
- WS - Win Shares
- WS/48 - Win Shares Per 48 Minutes
- blank2
- OBPM - Offensive Box Plus/Minus
- DBPM - Defensive Box Plus/Minus
- BPM - Box Plus/Minus
- VORP - Value Over Replacement
- FG - Field Goals
- FGA - Field Goal Attempts
- FG% - Field Goal Percentage
- 3P - 3-Point Field Goals
- 3PA - 3-Point Field Goal Attempts
- 3P% - 3-Point Field Goal Percentage
- 2P - 2-Point Field Goals
- 2PA - 2-Point Field Goal Attempts
- 2P% - 2-Point Field Goal Percentage
- eFG% - Effective Field Goal Percentage
- FT - Free Throws
- FTA - Free Throw Attempts
- FT% - Free Throw Percentage
- ORB - Offensive Rebounds
- DRB - Defensive Rebounds
- TRB - Total Rebounds
- AST - Assists
- STL - Steals
- BLK - Blocks
- TOV - Turnovers
- PF - Personal Fouls
- PTS - Points


##### Loading libraries and datasets


In [215]:
# Importing libraries
import pandas as pd
import featuretools as ft
import matplotlib.pyplot as plt
import plotly.express as px

In [229]:
# Loading datasets
player_data = pd.read_csv("data/player_data.csv")
players = pd.read_csv("data/players.csv")
seasons_stats = pd.read_csv("data/seasons_stats.csv")

##### Cleaning players data


In [230]:
# Cleaning players datasets
player_data = player_data.sort_values("name").drop_duplicates().reset_index(drop=True)
player_data[["birth_date", "birth_year"]] = player_data["birth_date"].str.split(
    ",", expand=True
)
players = (
    players.sort_values("Player")
    .drop_duplicates()
    .drop(columns=["Unnamed: 0", "born"])
    .reset_index(drop=True)
)

In [231]:
# Mergin players datasets
players_data = pd.merge(
    player_data, players, left_on="name", right_on="Player", how="inner"
).sort_values(["name", "birth_year"])

# Fill college values where possible
players_data["college"] = players_data["college"].fillna(players_data["collage"])

# Drop unnecessary columns
players_data = players_data.drop(
    columns=["collage", "height_x", "weight_x", "Player", "birth_city", "birth_state"]
)

# Drop rows with no college, birth_date, and position valuse
players_data = players_data.dropna(subset=["college", "birth_date", "position"])

# Generate new id column
players_data = players_data.reset_index(drop=True)
players_data["id"] = players_data.index

# Rename columns for convenience
players_data = players_data.rename(
    columns={"height_y": "height_cm", "weight_y": "weight_kg"}
)

# Rearrange columns for convenience
clean_players_data = players_data[
    [
        "id",
        "name",
        "birth_date",
        "birth_year",
        "height_cm",
        "weight_kg",
        "college",
        "year_start",
        "year_end",
        "position",
    ]
]

In [232]:
clean_players_data

,id,name,birth_date,birth_year,height_cm,weight_kg,college,year_start,year_end,position
0,0,A.C. Green,October 4,1963,203.0,106.0,Oregon State University,1986,2001,F-C
1,1,A.J. Bramlett,January 10,1977,196.0,88.0,University of Arizona,2000,2000,C
2,2,A.J. English,July 11,1967,196.0,95.0,Virginia Union University,1991,1992,G
3,3,A.J. Guyton,February 12,1978,208.0,99.0,Indiana University,2001,2003,G
4,4,A.J. Hammons,August 27,1992,198.0,99.0,Purdue University,2017,2017,C
...,...,...,...,...,...,...,...,...,...,...
3540,3540,Zach Randolph,July 16,1981,206.0,117.0,Michigan State University,2002,2018,F-C
3541,3541,Zaid Abdul-Aziz,April 7,1946,206.0,106.0,Iowa State University,1969,1978,C-F
3542,3542,Zeke Sinicola,January 25,1929,178.0,74.0,Niagara University,1952,1954,G
3543,3543,Zeke Zawoluk,October 13,1930,201.0,97.0,St. John's University,1953,1955,F-C


##### Cleaning statistics data


In [220]:
# # Cleaning stats data
seasons_stats = (
    seasons_stats.sort_values(["Player", "Year"])
    .drop(columns=["Unnamed: 0", "blanl", "blank2"])
    .dropna(subset=["Player"])
    .reset_index(drop=True)
)

# Remove stats for players that are absent in the
# players dataset
seasons_stats = seasons_stats[seasons_stats["Player"].isin(clean_players_data["name"])]

# Fill NaN values with 0
seasons_stats = seasons_stats.fillna(0)

# Fix numeric datatypes
seasons_stats["Year"] = seasons_stats["Year"].astype(int)
seasons_stats["Age"] = seasons_stats["Age"].astype(int)
seasons_stats["G"] = seasons_stats["G"].astype(int)


clean_season_statistics = seasons_stats

clean_season_statistics

,Year,Player,Pos,Age,Tm,G,GS,MP,PER,TS%,...,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS
0,1986,A.C. Green,PF,22,LAL,82,1.0,1542.0,11.8,0.564,...,0.611,160.0,221.0,381.0,54.0,49.0,49.0,99.0,229.0,521.0
1,1987,A.C. Green,PF,23,LAL,79,72.0,2240.0,15.7,0.599,...,0.780,210.0,405.0,615.0,84.0,70.0,80.0,102.0,171.0,852.0
2,1988,A.C. Green,PF,24,LAL,82,64.0,2636.0,14.5,0.581,...,0.773,245.0,465.0,710.0,93.0,87.0,45.0,120.0,204.0,937.0
3,1989,A.C. Green,PF,25,LAL,82,82.0,2510.0,17.8,0.594,...,0.786,258.0,481.0,739.0,103.0,94.0,55.0,119.0,172.0,1088.0
4,1990,A.C. Green,PF,26,LAL,82,82.0,2709.0,14.7,0.548,...,0.751,262.0,450.0,712.0,90.0,66.0,50.0,116.0,207.0,1061.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24600,2004,Zendon Hamilton,C,28,PHI,46,0.0,473.0,15.9,0.616,...,0.698,49.0,97.0,146.0,13.0,8.0,8.0,27.0,74.0,169.0
24601,2005,Zendon Hamilton,C,29,MIL,16,0.0,159.0,10.4,0.480,...,0.604,18.0,24.0,42.0,6.0,5.0,2.0,12.0,26.0,51.0
24602,2006,Zendon Hamilton,C,30,TOT,12,0.0,49.0,17.6,0.593,...,0.667,3.0,8.0,11.0,0.0,4.0,0.0,6.0,5.0,26.0
24603,2006,Zendon Hamilton,C,30,CLE,11,0.0,46.0,18.2,0.624,...,0.688,3.0,8.0,11.0,0.0,3.0,0.0,6.0,4.0,25.0


##### Extracting features for players' dataset from season statistics dataset


In [233]:
# 1. Extract features from the season statistics
new_player_features_from_season_statistics = clean_season_statistics.groupby(
    "Player", as_index=False
).aggregate(
    {
        "G": "sum",
        "MP": "sum",
        "PTS": "sum",
        "FG": "sum",
        "FGA": "sum",
        "FT": "sum",
        "3P": "mean",
        "PER": "mean",
        "WS": "sum",
    }
)

# 2. Merge new features into the players dataset
clean_players_data = pd.merge(
    clean_players_data,
    new_player_features_from_season_statistics,
    left_on="name",
    right_on="Player",
).drop(columns="Player")

# 3. Rename columns for convenience
clean_players_data = clean_players_data.rename(
    columns={
        "G": "games_played",
        "MP": "minutes_played",
        "PTS": "points_earned",
        "3P": "average_threepointers",
        "PER": "average_efficiency_rating",
        "FG": "field_goals",
        "FGA": "field_goal_attempts",
        "FT": "free_throws",
        "WS": "win_shares",
    }
)

##### Plotting datasets


In [222]:
# 1. Select a sample of players
players_sorted_by_college_and_birth_year = clean_players_data.sort_values(
    ["college", "birth_year"]
)
first_oldest_payers_in_college = (
    players_sorted_by_college_and_birth_year.groupby("college")
    .first()
    .sort_values("name")
    .reset_index()
)

# 2. Get first 20 players that played the most games
first_twenty_players_with_most_games_played = (
    first_oldest_payers_in_college.sort_values(["games_played"], ascending=False).head(
        20
    )
).reset_index(drop=True)

# 4. Build new index to avoid name overlapping
first_twenty_players_with_most_games_played["id"] = (
    first_twenty_players_with_most_games_played["name"]
    + " ("
    + first_twenty_players_with_most_games_played["birth_year"]
    + ")"
)


# 4. Plot a bar chart
fig = px.bar(
    first_twenty_players_with_most_games_played,
    x="id",
    y="games_played",
    color="college",
)
fig.show()

In [223]:
# 1. Select a sample of the most experienced players per college
top_twenty_most_experienced_players_per_college = (
    clean_players_data.sort_values(
        ["college", "minutes_played"], ascending=[True, False]
    )
    .groupby("college")
    .first()
    .reset_index()
    .sort_values("minutes_played", ascending=False)
    .head(20)
)

top_twenty_most_experienced_players_per_college

# 2. Filter season statistics to contain only rows for the most
# experiences players per college
season_statistics_of_most_experienced_players_per_college = clean_season_statistics[
    clean_season_statistics["Player"].isin(
        top_twenty_most_experienced_players_per_college["name"]
    )
]

# 3. Plot points earned by each player per year
fig = px.line(
    season_statistics_of_most_experienced_players_per_college,
    x="Year",
    y="PTS",
    color="Player",
)
fig.show()

In [224]:
# 1. Plot scatter graph when X axis represents games played and Y axis points
fig = px.scatter(
    clean_players_data,
    x="games_played",
    y="points_earned",
    color="name",
)
fig.show()

In [225]:
# 1. Generate name vs college scatter plot
fig = px.scatter(clean_players_data, x="name", y="college")
fig.show()

In [226]:
# I would like to check if a height of a palyer affects the amount of scored 3 point goals

# 1. Get a simple random sample of players
sample_of_players = clean_players_data.sample(n=500)

# 2. Plot scatter chart where X is the player's height and Y average threepointers
fig = px.scatter(
    sample_of_players, x="height_cm", y="average_threepointers", color="name"
)
fig.show()

In [234]:
# 1. Let's sum win shares for each college
sum_of_win_shares_per_college = (
    clean_players_data.groupby("college", as_index=False)
    .aggregate({"win_shares": "sum"})
    .sort_values(by="win_shares", ascending=False)
    .head(20)
)

# 4. Plot bar chart where X represents college and Y win shares
fig = px.bar(sum_of_win_shares_per_college, x="college", y="win_shares")
fig.show()

In [ ]:
# 1. Compute correlation matrix
players_quantitative_data = clean_players_data[
    [
        "birth_year",
        "height_cm",
        "weight_kg",
        "minutes_played",
        "win_shares",
        "games_played",
        "points_earned",
        "field_goals",
        "average_threepointers",
        "average_efficiency_rating",
    ]
]
players_correlation_matrix = players_quantitative_data.corr()

# 2. Plot heatmap representing correlation matrix
fig = px.imshow(
    players_correlation_matrix,
    text_auto=".2f",
)
fig.show()

In [239]:
# 1. Aggregate points earned and average PER per college
points_earned_per_college = clean_players_data.groupby(
    "college", as_index=False
).aggregate({"points_earned": "sum", "average_efficiency_rating": "mean"})

# 2. Plot scatter graph where X shows points earned and Y average PER
fig = px.scatter(
    points_earned_per_college,
    x="points_earned",
    y="average_efficiency_rating",
    color="college",
)
fig.show()

In [242]:
# 1. Plot scatter graph where X shows minutes played and Y average threepointers
fig = px.scatter(
    clean_players_data,
    x="minutes_played",
    y="average_threepointers",
    color="name",
)
fig.show()

In [244]:
# 1. Plot scatter graph where X shows minutes played and Y average threepointers
fig = px.scatter(
    clean_players_data,
    x="free_throws",
    y="average_threepointers",
    color="name",
)
fig.show()